# Modelling length of stay in the ICU

Mean of days spent in the ICU: 4.67\
Variance: 66.89\
=> overdispersion

## Frequentist approach

### Composite likelihood 

#### Regular length of stay in the ICU

Let's consider:
- Y: the random variable to explain, representing the number of days spent in the ICU , $Y \in \{0,28\}$
- X: the random variable representing the treatment group (0 for placebo) $X \in \{0,1\}$

PS: We do not consider the pateints who weren't admitted in the ICU whose length of stay is 0 (30/120 patients)

$Y \mid X = x \sim \text{BN}(\mu_x,\theta)$, $x \in \{0,1\}$

where $\mu_x$ is the BN parameter that depends on the value of $X$. Typically, in an exponential model:

$$
\mu_x = \exp(\beta_0 + \beta_1 x)
$$

We consider a sample ${\{(Y_1, X_1), \ldots, (Y_n, X_n)\}}$ drawn from ${(Y, X)}$.
The associated statistical model is given by
$$
\Bigl( (\mathbb{R}_+^*)^2n, \quad \mathcal{B}((\mathbb{R}_+^*)^2n), \quad \{ \text{BN}(\mu_x, \theta ), \ \mu_x, \theta > 0 \} \Bigr).
$$

The model is dominated by the counting measure $\delta$, and let $P_{\mu_x,\theta}$ be the probability distribution associated with the model. Then, the density with respect to $\delta$ is given by:

$$
\frac{dP_{\mu_x, \theta}}{d\delta}(y) = \frac{\Gamma(y + \theta)}{y! \ \Gamma(\theta)} \left(\frac{\theta}{\theta + \mu_x}\right)^{\theta} \left(\frac{\mu_x}{\theta + \mu_x}\right)^y
$$
where $y \in \mathbb{N}$, $\mu_x > 0$ is the mean parameter and $\theta > 0$ is the dispersion parameter, for $y \in \mathbb{N}$.\

a) The support of the density function does not depend on $\mu$ nor $\theta$, and the parameter space is open.\
b) The density function is $\mathcal{C}^\infty$ (infinitely differentiable) on the interval $]0, +\infty[^2$.

The likelihood is given by:

$$
L(\boldsymbol{\beta}) = \prod_{i=1}^n \left( \frac{\Gamma(Y_i + \theta)}{\Gamma(\theta) \, Y_i!} \left( \frac{\theta}{\theta + \mu_{X_i}} \right)^{\theta} \left( \frac{\mu_{X_i}}{\theta + \mu_{X_i}} \right)^{Y_i} \cdot \mathbf{1}_{\mathbb{N}}(Y_i) \cdot \mathbf{1}_{\{\mu_{X_i} > 0, \theta > 0\}} \right)
$$

with $\mu_{X_i} = \exp(\beta_0 + \beta_1 X_i)$.

The log-likelihood is:

$$
\ell(\boldsymbol{\beta}, \theta) = \sum_{i=1}^n \Biggl[ 
\log \Gamma(Y_i + \theta) - \log \Gamma(\theta) - \log (Y_i!) + \theta \log \left(\frac{\theta}{\theta + \mu_{X_i}}\right) + Y_i \log \left(\frac{\mu_{X_i}}{\theta + \mu_{X_i}}\right)
\Biggr]
$$

$$
\ell(\boldsymbol{\beta}) = \sum_{i=1}^n \log \Gamma(Y_i + \theta) - \sum_{i=1}^n \log (Y_i!) - n \log \Gamma(\theta) + \sum_{i=1}^n Y_i \left( \beta_0 + \beta_1 X_i - \log\bigl( e^{\beta_0 + \beta_1 X_i} + \theta \bigr) \right) + n \theta \log(\theta) - \sum_{i=1}^n \log \bigl( e^{\beta_0 + \beta_1 X_i} + \theta \bigr)
$$

Resolution in R

```
ll_BN<-function(b){
    b0 <- b[1]
    b1 <- b[2]
    mu <- exp(b0+b1*X)
    theta<- mean(Y,na.rm=TRUE)^2/(var(Y, na.rm = TRUE)-mean(Y,na.rm=TRUE))
    ll<-sum(
      lgamma(Y+theta)-
      lgamma(theta)-
      lgamma(Y+1)+
      theta*log(theta/(theta+mu))+
      Y*log(mu/(theta+mu))
    )
  return(ll)
}
result <- maxLik(logLik = ll_BN, start = c(0, 0))
summary(result)
print(paste("Average stay of placebo group:", exp(2.0332)))
print(paste("Average stay of treatment group:", exp(2.0332-0.3547)))
```

Results

```
--------------------------------------------
Maximum Likelihood estimation
Newton-Raphson maximisation, 5 iterations
Return code 8: successive function values within relative tolerance limit (reltol)
Log-Likelihood: -254.6944 
2  free parameters
Estimates:
     Estimate Std. error t value Pr(> t)    
[1,]   2.0332     0.1942  10.470  <2e-16 ***
[2,]  -0.3547     0.2850  -1.245   0.213    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
--------------------------------------------
```
**[1] "Average stay of placebo group: 7.63849046086658"**\
**[1] "Average stay of treatment group: 5.35751367039876**"


**These results are consitent with the descriptive statistics and with those of the NB regression**:
```
nb_model_ <- with(imp,glm.nb(EvolDiasUCI_tronq ~Grupo))
summary(pool(nb_model_), conf.int=TRUE)
```

| Term         | Estimate   | Std. Error | Statistic |    df    |   p-value    | 2.5 %     | 97.5 %    |
|--------------|------------|------------|-----------|----------|--------------|-----------|-----------|
| (Intercept)  | 1.8057849  | 0.2500329  | 7.222190  | 115.0385 | 5.959091e-11 | 1.310520  | 2.3010502 |
| Grupo1       | -0.4840291 | 0.3545368  | -1.365244 | 115.0385 | 1.748403e-01 | -1.186296 | 0.2182375 |

**[1] "Average stay of placebo group: 6.08474549143833"**\
**[1] "Average stay of treatment group: 3.74999985006631"**

#### Truncated length of stay in the ICU

$$
\text{truncated length of stay} =
\begin{cases}
\text{regular length of stay}, & \text{if the patient didn't die after 28 days} \\
28 & \text{if regular length of stay >28 or if the pateint died}

\end{cases}
$$

We now consider the truncated length of stay in the ICU and Z the variable which indicates if the patient died, $ Z \in \{0,1\}$. X,Y and Z are not independent.

$P(Y,Z/X)=\frac{P(Y,Z,X)}{P(X)}$

$P(Y/Z,X)=\frac{P(Y,Z,X)}{P(Z,X)}$ and $P(Z,X)= P(Z/X)P(X)$, so $P(Y,Z,X)=P(Y/Z,X)P(Z/X)P(X)$\
Therefore, $ P(Y,Z/X)=P(Y/Z,X)P(Z/X) $


$Y \mid X=x, Z = z \sim \text{BN}(\mu_{x,z},\theta)$, $x, z \in \{0,1\}$

where $\mu_{x,z}$ is the BN parameter that depends on the value of $X$:

$$
\mu_{x,z} = \exp(\beta_0 + \beta_1 x + \beta_2 z)
$$
So the density is :
$$
f_{Y \mid X,Z}(y\mid x,z) = \frac{\Gamma(y + \theta)}{y! \ \Gamma(\theta)} \left(\frac{\theta}{\theta + \mu_{x,z}}\right)^{\theta} \left(\frac{\mu_{x,z}}{\theta + \mu_{x,z}}\right)^y \cdot \mathbf{1}_{\mathbb{N}}(y) \cdot \mathbf{1}_\{{\mu_{x, z} > 0, \theta > 0\}}
$$
As $ Z \in \{0,1\}$ we model using a logistic regression:
$$
P(Z=z/X=x)= P(Z=1/X=x)^z \cdot P(Z=0/X=x)^{1-z} = \left(\frac{exp(\alpha_0 + \alpha_1 x)}{1+ exp(\alpha_0 + \alpha_1x)}\right)^z \left(\frac{1}{1+ exp(\alpha_0 + \alpha_1x)}\right)^{1-z}
$$
with $x, z \in \{0,1\}$

Finally we have:
$$
f_{Y,Z \mid X}(y, z \mid x) = \frac{\Gamma(y + \theta)}{y! \ \Gamma(\theta)} \left(\frac{\theta}{\theta + \mu_{x,z}}\right)^{\theta} \left(\frac{\mu_{x,z}}{\theta + \mu_{x,z}}\right)^y \left(\frac{exp(\alpha_0 + \alpha_1 x)}{1+ exp(\alpha_0 + \alpha_1x)}\right)^z \left(\frac{1}{1+ exp(\alpha_0 + \alpha_1x)}\right)^{1-z}\cdot \mathbf{1}_{\mathbb{N}}(y) \cdot \mathbf{1}_{\{\mu_{x, z} > 0, \theta > 0\}} \cdot \mathbf{1}_{x, z \in \{0,1\}}
$$

We consider a sample ${\{(Y_1, X_1, Z_1), \ldots, (Y_n, X_n, Z_,)\}}$ drawn from ${(Y, X, Z)}$.\
The model is dominated by the counting measure $\delta$, and let $P_{\mu_{x,z},\theta}$ be the probability distribution associated with the model. Then, the density with respect to $\delta$ is given by:

$$
\frac{dP_{\mu_{x,z}, \theta}}{d\delta}(y) = f_{Y,Z \mid X}(y, z \mid x)
$$

a) The support of the density function does not depend on $\mu$ nor $\theta$, and the parameter space is open.\
b) The density function is $\mathcal{C}^\infty$ (infinitely differentiable) on the interval $]0, +\infty[^2$.

The likelihood is given by:

$$
L(\boldsymbol{\mu, \theta}) = \prod_{i=1}^n  \frac{\Gamma(Y_i + \theta)}{\Gamma(\theta) \, Y_i!} \left( \frac{\theta}{\theta + \mu_{X_i, Z_i}} \right)^{\theta} \left( \frac{\mu_{X_i, Z_i}}{\theta + \mu_{X_i, Z_i}} \right)^{Y_i} \left(\frac{exp(\alpha_0 + \alpha_1 X_i)}{1+ exp(\alpha_0 + \alpha_1X_i)}\right)^{Z_i} \left(\frac{1}{1+ exp(\alpha_0 + \alpha_1X_i)}\right)^{1-Z_i} \cdot \mathbf{1}_{\mathbb{N}}(Y_i) \cdot \mathbf{1}_{\{\mu_{X_i, Z_i} > 0, \theta > 0\} \cdot \mathbf{1}_{X_i, Z_i \in \{0,1\}}} 
$$

with $\mu_{X_i, Z_i} = \exp(\beta_0 + \beta_1 X_i + + \beta_2 Z_i)$.

The log-likelihood is:

$$
\ell(\boldsymbol{\mu}, \theta) = \sum_{i=1}^n \Biggl[ 
\log \Gamma(Y_i + \theta) - \log (Y_i!) + \log \left(\frac{\theta^{\theta}}{\Gamma(\theta)}\right)- (\theta + Y_i) \log (\theta +\mu_{X_i, Z_i}) + Y_i \log (\mu_{X_i, Z_i}) + Z_i (\alpha_0 +\alpha_1 X_i) - log(1 + exp(\alpha_0 +\alpha_1 X_i))
\Biggr]
$$

$$
\ell(\boldsymbol{\mu}, \theta) = \sum_{i=1}^n \Biggl[ 
\log \Gamma(Y_i + \theta) - \log (Y_i!) + \log \left(\frac{\theta^{ \theta}}{\Gamma(\theta)}\right)- (\theta + Y_i) \log (\theta +exp(\beta_0 + \beta_1X_i +\beta_2Z_i)) + Y_i (\beta_0 + \beta_1X_i +\beta_2Z_i) + Z_i (\alpha_0 +\alpha_1 X_i) - log(1 + exp(\alpha_0 +\alpha_1 X_i))
\Biggr]
$$


#### **In particular**:

If Z=1, Y=28. 
- $Y \mid X=x, Z = 0 \sim \text{BN}(\mu_{x},\theta)$, $x \in \{0,1\}$ with $\mu_{x} = \exp(\beta_0 + \beta_1 x)$
- $P(Y=28,Z=1/X)=P(Z=1/X)=\left(\frac{exp(\alpha_0 + \alpha_1 x)}{1+ exp(\alpha_0 + \alpha_1x)}\right) \cdot \mathbf{1}_{x \in \{0,1\}}$
So:
$$
f_{Y \mid X,Z}(y\mid x,z)=f_{Y \mid X,Z=0}(y\mid x,z) \cdot \mathbf{1}_{z=0} + f_{Z=1 \mid X}(z\mid x) \cdot \mathbf{1}_{z=1} 
$$

Finally we have: 
$$
f_{Y \mid X,Z}(y\mid x,z)= 
C
$$

We consider a sample ${\{(Y_1, X_1, Z_1), \ldots, (Y_n, X_n, Z_,)\}}$ drawn from ${(Y, X, Z)}$.\
The model is dominated by the counting measure $\delta$, and let $P_{\mu_{x,z},\theta}$ be the probability distribution associated with the model. Then, the density with respect to $\delta$ is given by:

$$
\frac{dP_{\mu_{x,z}, \theta}}{d\delta}(y) = f_{Y,Z \mid X}(y, z \mid x)
$$

a) The support of the density function does not depend on $\mu$ nor $\theta$, and the parameter space is open.\
b) The density function is $\mathcal{C}^\infty$ (infinitely differentiable) on the interval $]0, +\infty[^2$.

The likelihood is given by:

$$
L(\boldsymbol{\mu, \theta}) = 
\begin{cases}

\prod_{i=1}^n  \frac{\Gamma(Y_i + \theta)}{\Gamma(\theta) \, Y_i!} \left( \frac{\theta}{\theta + \mu_{X_i}} \right)^{\theta} \left( \frac{\mu_{X_i}}{\theta + \mu_{X_i}} \right)^{Y_i}\left(\frac{1}{1+ exp(\alpha_0 + \alpha_1X_i)}\right) \cdot \mathbf{1}_{\mathbb{N}}(Y_i) \cdot \mathbf{1}_{\{\mu_{X_i} > 0, \theta > 0\} \cdot \mathbf{1}_{X_i \in \{0,1\}}} , & \text{for all i such that $Z_i$ =0} \\ 
\prod_{i=1}^n  \left(\frac{exp(\alpha_0 + \alpha_1X_i)}{1+ exp(\alpha_0 + \alpha_1X_i)}\right) \cdot \mathbf{1}_{X_i \in \{0,1\}}& \text{for all i such that $Z_i$ =1}

\end{cases}
$$

with $\mu_{X_i} = \exp(\beta_0 + \beta_1 X_i)$.

The log-likelihood is:

$$
\ell(\boldsymbol{\mu}, \theta) =
\begin{cases}

 \sum_{i=1}^n \Biggl[ 
\log \Gamma(Y_i + \theta) - \log (Y_i!) + \log \left(\frac{\theta^{\theta}}{\Gamma(\theta)}\right)- (\theta + Y_i) \log (\theta +\mu_{X_i}) + Y_i \log (\mu_{X_i}) - log(1 + exp(\alpha_0 +\alpha_1 X_i))
\Biggr] , & \text{for all i such that $Z_i$ =0} \\
\sum_{i=1}^n \Biggl[ 
(\alpha_0 +\alpha_1 X_i) - log(1 + exp(\alpha_0 +\alpha_1 X_i))
\Biggr] & \text{for all i such that $Z_i$ =1}

\end{cases}
$$

$$
\ell(\boldsymbol{\alpha, \beta}, \theta) =
\begin{cases}

 \sum_{i=1}^n \Biggl[ 
\log \Gamma(Y_i + \theta) - \log (Y_i!) + \log \left(\frac{\theta^{\theta}}{\Gamma(\theta)}\right)- (\theta + Y_i) \log (\theta +\exp(\beta_0 + \beta_1 X_i)) + Y_i\exp(\beta_0 + \beta_1 X_i) - log(1 + exp(\alpha_0 +\alpha_1 X_i))
\Biggr] , & \text{for all i such that $Z_i$ =0} \\
\sum_{i=1}^n \Biggl[ 
(\alpha_0 +\alpha_1 X_i) - log(1 + exp(\alpha_0 +\alpha_1 X_i))
\Biggr] & \text{for all i such that $Z_i$ =1}

\end{cases}
$$







Resolution in R
```
ll_NB_death<-function(params){
    a0 <- params[1]
    a1 <- params[2]
    b0 <- params[3]
    b1 <- params[4]
    X0<-as.numeric(as.character(data_dint$Grupo[!is.na(data_dint$EvolDiasUCI_tronq) & data_dint$EvolMort28==0 & data_dint$UCI_Interm==1]))
    X1<-as.numeric(as.character(data_dint$Grupo[!is.na(data_dint$EvolDiasUCI_tronq) & data_dint$EvolMort28==1 & data_dint$UCI_Interm==1]))
    Y0<-as.numeric(as.character(data_dint$EvolDiasUCI_tronq[!is.na(data_dint$EvolDiasUCI_tronq) & data_dint$EvolMort28==0 & data_dint$UCI_Interm==1]))
    mu <- exp(b0+b1*X0)
    theta<- mean(data_dint$EvolDiasUCI_tronq,na.rm=TRUE)^2/(var(data_dint$EvolDiasUCI_tronq, na.rm = TRUE)-mean(data_dint$EvolDiasUCI_tronq,na.rm=TRUE))
    ll_0<-sum(
      lgamma(Y0+theta)-
      lgamma(Y0+1)+
      theta* log(theta)-
      lgamma(theta)-
      (theta + Y0)*log(theta+mu)+
      Y0*log(mu)-
      log(1 + exp(a0+a1*X0))
    )
    ll_1<-sum(
      a0+a1*X1-
      log(1 + exp(a0+a1*X1))
    )
  ll <- ifelse(Z == 0, ll_0, ll_1) 
  return(ll)
}
Z<-as.numeric(as.character(data_dint$EvolMort28[!is.na(data_dint$EvolDiasUCI_tronq) & data_dint$UCI_Interm==1]))
result <- maxLik(logLik = ll_NB_death, start = c(0,0,0,0))
summary(result)
a0<--4.23669 
a1<--1.07425    
b0<-1.53870
b1<--0.09768
print(paste("Probaility of death in placebo group:",round((exp(a0) / (1 + exp(a0)))*100,2), "%"))
print(paste("Probaility of death in treatment group:",round((exp(a0+a1) / (1 + exp(a0+a1)))*100,2), "%"))
print(paste("Number of days in ICU for non dead patients in placebo group:", exp(b0)))
print(paste("Number of days in ICU for non dead patients in treatment group:", exp(b0+b1)))
--------------------------------------------
Maximum Likelihood estimation
Newton-Raphson maximisation, 8 iterations
Return code 8: successive function values within relative tolerance limit (reltol)
Log-Likelihood: -17277.85 
4  free parameters
Estimates:
     Estimate Std. error t value  Pr(> t)    
[1,] -4.23669    0.14141 -29.959  < 2e-16 ***
[2,] -1.07425    0.28291  -3.797 0.000146 ***
[3,]  1.53870    0.02873  53.550  < 2e-16 ***
[4,] -0.09768    0.04102  -2.381 0.017264 *  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
--------------------------------------------
```

**[1] "Probability of death in placebo group: 1.42 %"**\
**[1] "Probaility of death in treatment group: 0.49 %"**\
**[1] "Number of days in ICU for non dead patients in placebo group: 4.65853024350716"**\
**[1] "Number of days in ICU for non dead patients in treatment group: 4.22500312308255"**\

**These results are consitent with the descriptive statistics and with those of the NB regression**:
```
nb_model <- with(
  imp,
  glm.nb(EvolDiasUCI_tronq ~ Grupo + EvolMort28, subset = UCI_Interm == 1)
)
summary(pool(nb_model))
```
| Term         | Estimate    | Std. Error | Statistic  |    df    |   p-value    |
|--------------|-------------|------------|------------|----------|--------------|
| (Intercept)  | 1.53492137  | 0.1936029  | 7.9281926  | 84.05899 | 8.409724e-12 |
| Grupo1       | -0.08996325 | 0.2706619  | -0.3323823 | 84.05899 | 7.404280e-01 |
| EvolMort281  | 1.82050411  | 0.4486811  | 4.0574565  | 84.05899 | 1.105896e-04 |

In a regular model, we can  construct asymptotic confidence intervals for $\theta$ based on the maximum likelihood estimator. Indeed, the maximum likelihood estimator $\hat{\theta}_n$ is asymptotically efficient, which means:

$$
\sqrt{n}(\hat{\theta}_n - \theta) \xrightarrow{\mathcal{L}} \mathcal{N} \left(0, \frac{1}{I(\theta)}\right)
$$

Hence, an asymptotic confidence interval of level $1 - \alpha$ is given by:

$$
\left[
\hat{\theta}_n - z_{1 - \alpha/2} \cdot \sqrt{\frac{1}{n I(\hat{\theta}_n)}}, \;
\hat{\theta}_n + z_{1 - \alpha/2} \cdot \sqrt{\frac{1}{n I(\hat{\theta}_n)}}
\right]
$$

where $z_{1 - \alpha/2}$ is the $(1 - \alpha/2)$ quantile of the standard normal distribution, and $I(\hat{\theta}_n)$ is the Fisher information evaluated at $\hat{\theta}_n$.


#### Ventilation-free days
Ventilation-free days=28 - truncated length of stay in the ICU

$$
\text{Ventilation-free days} =
\begin{cases}
28- \text{regular length of stay}, & \text{if the patient didn't die after 28 days} \\
0 & \text{if regular length of stay >28 or if the pateint died (structural zeros)}

\end{cases}
$$

- **Structural zeros : 8.89%**
- **Proportion of zeros for negative binomial model****:

$$
f_{Y \mid X,Z}(y=0\mid x,z)= 
\left(\frac{\theta}{\theta + \mu_{x}}\right)^{\theta} \left(\frac{1}{1+ exp(\alpha_0 + \alpha_1x)}\right)\cdot \mathbf{1}_{\{\mu_{x} > 0, \theta > 0\}} \cdot \mathbf{1}_{x \in \{0,1\}}, if z=0 
$$
**proportion= 1.66%**

=> zero-inflated

Let's consider:
- Y: the random variable to explain, representing the number of ventilation-free days, $Y \in \{0,28\}$
- X: the random variable representing the treatment group (0 for placebo) $X \in \{0,1\}$
- Z: the random variable which indicates if the patient died, $ Z \in \{0,1\}$

$$
f_{Y \mid X,Z}(y\mid x,z)= 
\begin{cases}
\Biggl[\pi \left(\frac{exp(\alpha_0 + \alpha_1 x)}{1+ exp(\alpha_0 + \alpha_1x)}\right)  + (1-\pi)\left(\frac{\theta}{\theta + \mu_{x}}\right)^{\theta} \left(\frac{1}{1+ exp(\alpha_0 + \alpha_1x)}\right)\Biggl]\cdot \mathbf{1}_{\{\mu_{x} > 0, \theta > 0\}} \cdot \mathbf{1}_{x \in \{0,1\}} , & \text{if y=0}    \\
(1-\pi)\frac{\Gamma(y + \theta)}{y! \ \Gamma(\theta)} \left(\frac{\theta}{\theta + \mu_{x}}\right)^{\theta} \left(\frac{\mu_{x}}{\theta + \mu_{x}}\right)^y \left(\frac{1}{1+ exp(\alpha_0 + \alpha_1x)}\right)\cdot \mathbf{1}_{\mathbb{N}}(y) \cdot \mathbf{1}_{\{\mu_{x} > 0, \theta > 0\}} \cdot \mathbf{1}_{x \in \{0,1\}}, & \text{if y>0}
\end{cases}
$$

We consider a sample ${\{(Y_1, X_1, Z_1), \ldots, (Y_n, X_n, Z_,)\}}$ drawn from ${(Y, X, Z)}$.\

The likelihood is given by:

$$
L(\boldsymbol{\mu, \theta}) = 
\begin{cases}
\prod_{i=1}^n \Biggl[\pi \left(\frac{exp(\alpha_0 + \alpha_1 X_i)}{1+ exp(\alpha_0 + \alpha_1X_i)}\right)  + (1-\pi)\left(\frac{\theta}{\theta + \mu_{X_i}}\right)^{\theta} \left(\frac{1}{1+ exp(\alpha_0 + \alpha_1X_i)}\right)\Biggl]\cdot \mathbf{1}_{\{\mu_{X_i} > 0, \theta > 0\}} \cdot \mathbf{1}_{X_i \in \{0,1\}} , & \text{for all i such that  $Y_i$ =0 }\\
(1-\pi)^n\prod_{i=1}^n  \frac{\Gamma(Y_i + \theta)}{\Gamma(\theta) \, Y_i!} \left( \frac{\theta}{\theta + \mu_{X_i}} \right)^{\theta} \left( \frac{\mu_{X_i}}{\theta + \mu_{X_i}} \right)^{Y_i}\left(\frac{1}{1+ exp(\alpha_0 + \alpha_1X_i)}\right) \cdot \mathbf{1}_{\mathbb{N}}(Y_i) \cdot \mathbf{1}_{\{\mu_{X_i} > 0, \theta > 0\} \cdot \mathbf{1}_{X_i \in \{0,1\}}} , & \text{for all i such that $Y_i$ >0}

\end{cases}
$$

with $\mu_{X_i} = \exp(\beta_0 + \beta_1 X_i)$.

The log-likelihood is:

$$
\ell(\boldsymbol{\mu}, \theta) =
\begin{cases}
\sum_{i=1}^n 
\log \Biggl[\pi \left(\frac{exp(\alpha_0 + \alpha_1 X_i)}{1+ exp(\alpha_0 + \alpha_1X_i)}\right)  + (1-\pi)\left(\frac{\theta}{\theta + \mu_{X_i}}\right)^{\theta} \left(\frac{1}{1+ exp(\alpha_0 + \alpha_1X_i)}\right)\Biggl]\cdot \mathbf{1}_{\{\mu_{X_i} > 0, \theta > 0\}} \cdot \mathbf{1}_{X_i \in \{0,1\}} , & \text{for all i such that $Y_i$ =0 } \\
 n\log (1-\pi) + \sum_{i=1}^n \Biggl[ 
 \log \Gamma(Y_i + \theta) - \log (Y_i!) + \log \left(\frac{\theta^{\theta}}{\Gamma(\theta)}\right)- (\theta + Y_i) \log (\theta +\mu_{X_i}) + Y_i \log (\mu_{X_i}) - log(1 + exp(\alpha_0 +\alpha_1 X_i))
\Biggr] , & \text{for all i such that $Y_i$ >0} 

\end{cases}
$$

$$
\ell(\boldsymbol{\alpha, \beta}, \theta) =
\begin{cases}
\sum_{i=1}^n  
\log \Biggl[\pi \left(\frac{exp(\alpha_0 + \alpha_1 X_i)}{1+ exp(\alpha_0 + \alpha_1X_i)}\right)  + (1-\pi)\left(\frac{\theta}{\theta + \exp(\beta_0 + \beta_1 X_i)}\right)^{\theta} \left(\frac{1}{1+ exp(\alpha_0 + \alpha_1X_i)}\right)\Biggl]\cdot \mathbf{1}_{\{\theta > 0\}} \cdot \mathbf{1}_{X_i \in \{0,1\}}
 , & \text{for all i such that  $Y_i$ =0 } \\
 n\log (1-\pi) + \sum_{i=1}^n \Biggl[ 
\log \Gamma(Y_i + \theta) - \log (Y_i!) + \log \left(\frac{\theta^{\theta}}{\Gamma(\theta)}\right)- (\theta + Y_i) \log (\theta +\exp(\beta_0 + \beta_1 X_i)) + Y_i\exp(\beta_0 + \beta_1 X_i) - log(1 + exp(\alpha_0 +\alpha_1 X_i))
\Biggr] , & \text{for all i such that $Y_i$ >0}

\end{cases}
$$

Resolution in R
```
ll_ZINB_death<-function(params){
    a0 <- params[1]
    a1 <- params[2]
    b0 <- params[3]
    b1 <- params[4]
    X0<-as.numeric(as.character(data_dint$Grupo[!is.na(data_dint$vf_days) & data_dint$vf_days==0  & data_dint$UCI_Interm==1]))
    X1<-as.numeric(as.character(data_dint$Grupo[!is.na(data_dint$vf_days)& data_dint$vf_days>0  & data_dint$UCI_Interm==1]))
    Y1<-as.numeric(as.character(data_dint$vf_days[!is.na(data_dint$vf_days)& data_dint$vf_days>0  & data_dint$UCI_Interm==1]))
    mu_0 <- exp(b0+b1*X0)
    mu_1 <- exp(b0+b1*X1)
    theta<- mean(data_dint$vf_days,na.rm=TRUE)^2/(var(data_dint$vf_days, na.rm = TRUE)-mean(data_dint$vf_days,na.rm=TRUE))
    ll_0<-sum(
      log(
        pst*(exp(a0+a1*X0)/(1+exp(a0+a1*X0)))+
        (1-pst)*((theta/(theta+mu_0))^theta)*(1/(1+exp(a0+a1*X0)))
      )
    )
    ll_1<-120*log (1-pst) + sum(
      lgamma(Y1+theta)-
      lgamma(Y1+1)+
      theta* log(theta)-
      lgamma(theta)-
      (theta + Y1)*log(theta+mu_1)+
      Y1*log(mu_1)-
      log(1 + exp(a0+a1*X1))
    )
  ll <- ifelse(Y==0, ll_0, ll_1)

  return(ll)
}
result <- maxLik(logLik = ll_ZINB_death, start = c(0,0,0,0))
summary(result)
print(paste("Probability of death in placebo group:",round((exp(-2.628338) / (1 + exp(-2.628338)))*100,2), "%"))
print(paste("Probaility of death in treatment group:",round((exp(-2.628338-0.822240) / (1 + exp(-2.628338-0.822240)))*100,2), "%"))
print(paste("Number of ventilation-free days for non dead patients in placebo group:", exp(3.174902)))
print(paste("Number of ventilation-free days for non dead patients in treatment group:", exp(3.174902+0.019032)))
--------------------------------------------
Maximum Likelihood estimation
Newton-Raphson maximisation, 7 iterations
Return code 8: successive function values within relative tolerance limit (reltol)
Log-Likelihood: -18458.93 
4  free parameters
Estimates:
      Estimate Std. error t value  Pr(> t)    
[1,] -2.628236   0.076514 -34.350  < 2e-16 ***
[2,] -0.822187   0.135645  -6.061 1.35e-09 ***
[3,]  3.174908   0.006955 456.469  < 2e-16 ***
[4,]  0.019031   0.009882   1.926   0.0541 .  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
--------------------------------------------
```
**[1] "Probability of death in placebo group: 6.73 %"**\
**[1] "Probaility of death in treatment group: 3.08 %"**\
**[1] "Number of ventilation-free days for non dead patients in placebo group: 23.9244751549209"**\
**[1] "Number of ventilation-free days for non dead patients in treatment group: 24.3841663115359"**

**The results are consistent with the descriptive statistics**



## Bayesian approach

### Modelling treatment failure

Let's consider:\
X: random variable representing the treatment group (0 for placebo)\
Y: random variable representing the treatment failure (1 for failure)\
$Y=y \mid X=x \sim \text{B}(p(x))$, $x,y \in \{0,1\}$ 
where $p(x)= P(Y=1/X=x) = \frac{exp(\beta_0 + \beta_1x)}{1+exp(\beta_0 + \beta_1x)}$\
According to Bayes formula:
$
\pi(\beta /y)\approx f(y/\beta)\pi(\beta)
$, 
with:
- $\beta =(\beta_0, \beta_1) \sim \text{N}(\mu, \Sigma)$ where $\Sigma=\binom{\sigma_0^{2}   \rho_{\sigma_0\sigma_1}}{\rho_{\sigma_0 \sigma_1}  \sigma_1^{2}}$ and $ \mu = \binom{\mu_0}{\mu_1}$
- $f(y/\beta)= \prod_{i=1}^n [\frac{exp(\beta_0+\beta_1x_i)^{y_i}}{1+exp(\beta_0+\beta_1x_i)}]$
- $ \pi(\beta)=\frac{exp(-\frac{1}{2}(\beta-\mu)^T\Sigma^-1(\beta-\mu))}{2\pi|\Sigma|^\frac{1}{2}}$ \
Considering a sample $(x,y)=((x_1,y_1),..., (x_n,y_n))$\
Finally:
$$
\pi(\beta/y)=\frac{exp(-\frac{1}{2}(\beta-\mu)^T\Sigma^-1(\beta-\mu))}{2\pi|\Sigma|^\frac{1}{2}}\prod_{i=1}^n [\frac{exp(\beta_0+\beta_1x_i)^{y_i}}{1+exp(\beta_0+\beta_1x_i)}]
$$

* **Non informative scenario**: \
$\beta_0$, $\beta_1 \sim \text{N}(0,10), \text{IC}_{95\%} = [-19.6, 19.6]$\
=> We have a large range so we're not sure if the treatment works or not

* **Optimistic scenario**: \
$\beta_0 \sim \text{N}(-0.8,0.2), \text{IC}_{95\%} = [-1.192, -0.408]$, IC for proportions : [0.23, 0.4]\
$\beta_1 \sim \text{N}(-0.4,0.2), \text{IC}_{95\%} = [-0.792, 0.008]$\
=> The treatment reduces the chaces of having a treatment failure by 0 to 45%. The proportion for the placebo group is close to the real one of 0.29

* **Pessimistic scenario**: \
$\beta_0 \sim \text{N}(-0.8,0.2)$\
$\beta_1 \sim \text{N}(0,0.2), \text{IC}_{95\%} = [-0.392, 0.392]$\
=> The treatment may have no effect (0 in the IC)


Resolution in R using jags:


```
#Non informative scenario
modelString="
model {
  for (i in 1:N) {
     Y[i] ~ dbern(p[i])
     Y_pred[i] ~ dbern(p[i])
     p[i] <- exp(beta0 + beta1*X[i])/(1+exp(beta0 + beta1*X[i]))
  }
  beta0 ~ dnorm(0,0.1)
  beta1 ~ dnorm(0,0.1)
} 
mod <- bmod$model(nchains = 3, nadapt = 1000, burnin = 50000)
samp <- bmod$sample(model = mod, variable_names = c("beta0", "beta1"), niter =100000, thin = 15)
bmod$metrics(samp, type_of_metric = "gelman")
Potential scale reduction factors:

      Point est. Upper C.I.
beta0          1          1
beta1          1          1

Multivariate psrf

1
```
**The Gelman-Rubin criterion (which compares the average intra-chain variance (W) with the inter-chain variance (B)) is equal to 1, so the 3 chains have converged towards the same distribution.**
```
bmod$metrics(samp, type_of_metric = "autocorr")
```
| Lag     | beta0        | beta1        |
|---------|--------------|--------------|
| Lag 0   | 1.000000000  | 1.000000000  |
| Lag 15  | 0.298828353  | 0.305783136  |
| Lag 75  | 0.003950814  | 0.007647593  |
| Lag 150 | -0.001621367 | -0.004266949 |
| Lag 750 | -0.014145748 | -0.010471767 |

**The autocorrelations are all <0.4 so the samples are not dependent (if samples are autocorrelated, they do not bring enough new information, so less autocorrelation means more information per sample. Autocorrelation can also slow down or even prevent convergence)**
```
bmod$metrics(samp, type_of_metric = "effectivesize")
```
beta0: 10 795\
beta1: 10 831\
**10 000 independent samples for each parameters, so estimations are reliable**
```
bmod$metrics(samp, type_of_metric = "effectivesize")

Iterations = 51015:150990
Thinning interval = 15 
Number of chains = 3 
Sample size per chain = 6666 

1. Empirical mean and standard deviation for each variable,
   plus standard error of the mean:

         Mean     SD Naive SE Time-series SE
beta0  0.2042 0.6832 0.004831       0.006578
beta1 -1.1429 0.4828 0.003414       0.004644

2. Quantiles for each variable:

        2.5%     25%     50%     75%  97.5%
beta0 -1.147 -0.2472  0.2049  0.6665  1.537
beta1 -2.111 -1.4567 -1.1308 -0.8125 -0.221
```
**In average, the tretament reduces the chances of treatment failure of  68% and there is a 97,5% chance of a reduction of at least 21%. Theses results are similar to those of the frequentist logistic regression**:
| Variable_to_explain | Odds_CI    | p_value  |
|---------------------|------------------------|------------|
| TreatmentFailureMan | 0.32 (0.12 - 0.85)     | 0.022941429|



### Modelling length of stay in the ICU

Let's consider:\
X: random variable representing the treatment group (0 for placebo)\
Y: random variable representing the number of days in the ICU

$Y=y \mid X=x \sim \text{BN}(p(x), \theta)$, $x,y \in \{0,1\}$ 
where :
- $E(Y)=\mu(x) = \theta\frac{1-p(x)}{p(x)}$ with $\mu(x)=exp(\beta_0 + \beta_1x)$, so $p(x)=\frac{\theta}{exp(\beta_0 + \beta_1x)+\theta}$\
According to Bayes formula:
$
\psi(\beta, \theta/y)\approx f(y/\beta, \theta)\pi(\beta)\phi(\theta)$ (we suppose $\theta$ and $\beta$ independant)
, 
with:
- $\beta =(\beta_0, \beta_1) \sim \text{N}(\mu, \Sigma)$ where $\Sigma=\binom{\sigma_0^{2}   \rho_{\sigma_0\sigma_1}}{\rho_{\sigma_0 \sigma_1}  \sigma_1^{2}}$ and $ \mu = \binom{\mu_0}{\mu_1}$
- $ \pi(\beta)=\frac{exp(-\frac{1}{2}(\beta-\mu)^T\Sigma^-1(\beta-\mu))}{2\pi|\Sigma|^\frac{1}{2}}$
- $f(y/\beta, \theta)= \prod_{i=1}^n [\binom{y_i + \theta - 1}{y_i} \, p(x_i)^\theta (1 - p(x_i))^{y_i}]=$
$\prod_{i=1}^n [\binom{y_i + \theta - 1}{y_i} \, (\frac{\theta}{exp(\beta_0 + \beta_1x_i)+\theta})^\theta (1 - \frac{\theta}{exp(\beta_0 + \beta_1x_i)+\theta})^{y_i}]$\
- We suppose $\theta \sim \Gamma(a,b)$:
$ \phi(\theta)= \theta^{a-1}\frac{b^a e{-b\theta}}{\Gamma{a}} $\
considering a sample $(x,y)=((x_1,y_1),..., (x_n,y_n))$\
Finally:
$$
\psi(\beta, \theta/y)\approx \theta^{a-1}\frac{b^a e{-b\theta}}{\Gamma{a}} \frac{exp(-\frac{1}{2}(\beta-\mu)^T\Sigma^-1(\beta-\mu))}{2\pi|\Sigma|^\frac{1}{2}} \prod_{i=1}^n [\binom{y_i + \theta - 1}{y_i} \, (\frac{\theta}{exp(\beta_0 + \beta_1x_i)+\theta})^\theta (1 - \frac{\theta}{exp(\beta_0 + \beta_1x_i)+\theta})^{y_i}]
$$
The average number of days in the ICU for non dead placebo patients is 4.66 days
* **Non informative scenario**: \
$\beta_1,\beta_0 \sim \text{N}(0,10), \text{IC}_{95\%} = [-19.6, 19.6]$\
=> We have a large range so we're not sure if the treatment works or not\
$\theta \sim \Gamma(0.01,0.01), \text{IC}_{95\%} = [3.5e-159, 4.7]$\
=> Very large range so no concrete information on the overdispersion

* **Optimistic scenario**: \
$\beta_0 \sim \text{N}(-0.8,0.2), \text{IC}_{95\%} = [-1.192, -0.408]$, IC for proportions : [0.23, 0.4]\
$\beta_1 \sim \text{N}(-0.4,0.2), \text{IC}_{95\%} = [-0.792, 0.008]$\
=> The treatment reduces the chaces of having a treatment failure by 0 to 45%. The proportion for the placebo group is close to the real one of 0.29\
$\theta \sim \Gamma(35,80), \text{IC}_{95\%} = [0.30, 0.59]$\
=> small variability of $\theta$ w around the true value of 0.35, meaning a strong overdispersion so a variable number of days in the ICU amongst patients

* **Pessimistic scenario**: \
$\beta_0 \sim \text{N}(-0.8,0.2)$\
$\beta_1 \sim \text{N}(0,0.2), \text{IC}_{95\%} = [-0.392, 0.392]$\
=> The treatment may have no effect (0 in the IC)\
$\theta \sim \Gamma(10,1), \text{IC}_{95\%} = [4.79, 17.09]$\
=> Moderate varaibility of $\theta$ but very far from the true value, with also a moderate overdispersion: the number of days in spent in the ICU do not vary as much among patients


Resolution in R using jags:
```
#non informative scenario
modelString_nb="
model {
  for (i in 1:N) {
     Y[i] ~ dnegbin(p[i], theta)
     Y_pred[i] ~ dnegbin(p[i], theta)
     p[i] <- theta/(exp(beta0 + beta1*X[i])+theta)
  }
  beta0 ~ dnorm(0,0.1)
  beta1 ~ dnorm(0,0.1)
  theta~ dgamma(0.01, 0.01)
} 
"
```
**Gelman-rubin score=1**

| Lag     | beta0        | beta1        | theta        |
|---------|--------------|--------------|--------------|
| Lag 0   | 1.000000000  | 1.000000000  | 1.000000000  |
| Lag 15  | 0.336893487  | 0.351558077  | -0.006570189 |
| Lag 75  | -0.006897206 | -0.010305502 | 0.003167876  |
| Lag 150 | -0.017999283 | -0.019692000 | 0.011369089  |
| Lag 750 | 0.002480239  | -0.001547056 | 0.006977153  |

**Autocorrelations <0.4**\
**effective sizes all >1000 : beta0 -> 2484, beta1 ->2571, theta -> 4998**

Results:
```
Iterations = 10015:34990
Thinning interval = 15 
Number of chains = 3 
Sample size per chain = 1666 

1. Empirical mean and standard deviation for each variable,
   plus standard error of the mean:

         Mean      SD Naive SE Time-series SE
beta0  2.3360 0.47121 0.006665       0.009516
beta1 -0.3108 0.30773 0.004353       0.006143
theta  0.5076 0.09301 0.001316       0.001316

2. Quantiles for each variable:

         2.5%     25%     50%     75%  97.5%
beta0  1.4212  2.0165  2.3372  2.6459 3.2999
beta1 -0.9004 -0.5137 -0.3185 -0.1057 0.3188
theta  0.3527  0.4404  0.5000  0.5645 0.7137
```
**Similar to frequentist negative binomial model**:

| Term        | Estimate   | Std. Error | Statistic |    df    |   p-value    | 2.5 %     | 97.5 %    |
|-------------|------------|------------|-----------|----------|--------------|-----------|-----------|
| (Intercept) | 1.8057849  | 0.2500329  | 7.222190  | 115.0385 | 5.959091e-11 | 1.310520  | 2.3010502 |
| Grupo1      | -0.4840291 | 0.3545368  | -1.365244 | 115.0385 | 1.748403e-01 | -1.186296 | 0.2182375 |
